# 03 — Build Shards (Stages 4–7)
**Does:** retrieval + cleaning + tokenization → bounded, validated, privacy-stripped shards for tracked periods AND the reference sample. This notebook is where raw text becomes training-ready tokens.
**Sources:** Arctic Shift paged search (`fields=`-trimmed, cursor-persisted) week by week; local `.zst`/`.jsonl` files when supplied (same output format). Per-subreddit download-tool `.jsonl` is the recommended Tier-1 bulk path — verify its URL pattern in the pilot cell before relying on it.
**Rules enforced:** exact-dedup within (sub, period) via capped hash set; language policy with recorded method (heuristic only if `ALLOW_HEURISTIC_LANG`, else a real detector is required); no usernames in shards (`rid_hash` only); overflow fractions from frozen periods; reference inclusion by stable-ID hash (order-independent, expandable).


In [ ]:
# Cell 1 — SHARD PLAN (the only cell you must edit).
PERIOD_FILTER = None     # e.g. ['w2v__askacademia__2019q1']; None = all sufficient tracked+reference periods
SUB_FILTER = None          # e.g. 'AskAcademia'; None = all
BUILD_TRACKED = True
BUILD_REFERENCE = True
ALLOW_HEURISTIC_LANG = True   # False = require a real language detector for production (recommended)
PAGES_PER_WEEK_CAP = None     # None = uncapped production; set e.g. 10 for a cheap proving run (labeled partial)
LOCAL_FILES = []              # e.g. ['/content/tmp_reddit/x_comments.zst']; streamed when non-empty
DRY_RUN = False               # True = synthetic records through the full writer+validator, offline
print(f"periods={PERIOD_FILTER} tracked={BUILD_TRACKED} ref={BUILD_REFERENCE} heur_lang={ALLOW_HEURISTIC_LANG}")


In [ ]:
# Cell 2 — Setup: root, config, cleaner, paths, manifest helpers.
import os, sys, csv, json, gzip, time, gc, hashlib, datetime
from pathlib import Path
from collections import defaultdict
import requests, yaml
from src.paths import get_project_root, resolve_tmp
from src.storage import atomic_write_text, sha256_file
from src.cleaner import clean_and_tokenize, extract_text, lang_of, _tests
from src.manifests import (RETRIEVAL_COLS, SHARD_COLS, load_manifest,
                           upsert_manifest_row, verify_output_shards)

ROOT = get_project_root()
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
ver = tuple(int(x) for x in cfg["config_version"].lstrip("v").split("."))
assert ver >= (0, 3, 0), f"need config v0.3.0+, found {cfg['config_version']}"
CFG_SHA = sha256_file(ROOT / "config/project_config.yaml")
SL = cfg["slicing"]; CL = cfg["cleaning"]
print("config", cfg["config_version"])

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/03_shards__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("sh"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

for d in ["shards/tokenized", "shards/temporary", "shards/quarantine", "diagnostics/shards"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

TMP = resolve_tmp(ROOT, cfg)
print("tmp:", TMP)

# Self-test cleaner from src.cleaner
for _raw, _expected in _tests:
    _res, _ = clean_and_tokenize(_raw)
    assert _res == _expected, "cleaner sanity test failed"
print("cleaner verified (imported from src.cleaner)")

RMAN = ROOT / "manifests/retrieval_manifest.csv"
if not RMAN.exists(): atomic_write_text(RMAN, ",".join(RETRIEVAL_COLS) + "\n")
def rrows(): return load_manifest(RMAN)
def rupsert(row): upsert_manifest_row(RMAN, row, ["unit_id"], RETRIEVAL_COLS)

SMAN = ROOT / "manifests/shard_manifest.csv"
if not SMAN.exists(): atomic_write_text(SMAN, ",".join(SHARD_COLS) + "\n")
def srows(): return load_manifest(SMAN)
def supsert(row): upsert_manifest_row(SMAN, row, ["shard_id"], SHARD_COLS)

def atomic_text(path: Path, text: str): atomic_write_text(path, text)
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
BASE = "https://arctic-shift.photon-reddit.com/api"
from src.api import api_get
print("setup ready")

In [ ]:
# Cell 3 — SHARD WRITER + API pager + local-file streamer (EOF-flushed, connection-closed).
class ShardWriter:
    """Bounded compressed shard: header line, JSON records, atomic validated close."""
    def __init__(self, shard_id, meta):
        self.id, self.meta, self.n, self.tok = shard_id, meta, 0, 0
        self.mints, self.maxts, self.units = None, None, set()
        self.final = ROOT / "shards/tokenized" / (shard_id + ".jsonl.gz")
        self.final.parent.mkdir(parents=True, exist_ok=True)
        self.tmp = self.final.with_suffix(".tmp")
        self.f = gzip.open(self.tmp, "wt", encoding="utf-8")
        self.f.write("#manifest " + json.dumps(meta) + "\n")
        self.hashes = set()
    def add(self, rec, unit_id):
        self.f.write(json.dumps(rec) + "\n"); self.units.add(unit_id)
        self.n += 1; self.tok += len(rec["tokens"])
        self.mints = rec["ts"] if self.mints is None else min(self.mints, rec["ts"])
        self.maxts = rec["ts"] if self.maxts is None else max(self.maxts, rec["ts"])
    def close(self):
        self.f.close(); os.replace(self.tmp, self.final)
        with gzip.open(self.final, "rt", encoding="utf-8") as f:
            head = f.readline(); body = sum(1 for _ in f)
        assert head.startswith("#manifest") and body == self.n, f"shard {self.id} reopen mismatch"
        sha = hashlib.sha256(open(self.final, "rb").read()).hexdigest()
        return {"shard_id": self.id, "n_records": self.n, "n_tokens": self.tok, "min_ts": self.mints,
                "max_ts": self.maxts, "path": str(self.final), "sha256": sha,
                "bytes_compressed": self.final.stat().st_size, "source_unit_ids": ";".join(sorted(self.units))}
EP = {"comments": "/comments/search", "submissions": "/posts/search"}
FIELDS = ",".join(cfg["counting"]["paged_fallback"]["fields"])
def api_pages(sub, ctype, after_u, before_u, unit_id):
    """Yields raw records oldest-first; persists cursor in manifest every page."""
    after, pages = after_u, 0
    while True:
        if PAGES_PER_WEEK_CAP is not None and pages >= PAGES_PER_WEEK_CAP: break
        r = api_get(EP[ctype], {"subreddit": sub, "after": after, "before": before_u, "limit": 100,
                                "sort": "asc", "fields": FIELDS}, tries=4)
        batch = r.json().get("data", [])
        if not batch: break
        pages += 1
        yield batch
        after = int(batch[-1].get("created_utc", after)) + 1
        if len(batch) < 100: break
def stream_local(path):
    """Record-by-record over .zst/.jsonl with decompressor flush + tail-line processing. Same schema out."""
    import zstandard as zstd
    p = Path(path)
    fh = gzip.open(p, "rt", encoding="utf-8") if p.suffix == ".gz" else open(p, "rb")
    try:
        if p.suffix == ".zst":
            dobj = zstd.ZstdDecompressor().decompressobj()
            buf = b""
            while True:
                chunk = fh.read(1 << 20) if not isinstance(fh, gzip.GzipFile) else fh.buffer.read(1 << 20)
                if not chunk: break
                try: buf += dobj.decompress(chunk)
                except Exception: continue
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    if line.strip():
                        try: yield json.loads(line)
                        except Exception: continue
            try:
                tail = dobj.flush()
                if tail: buf += tail
            except Exception: pass
            if buf.strip():
                try: yield json.loads(buf)
                except Exception: pass
        else:
            import io
            text = fh if isinstance(fh, gzip.GzipFile) or getattr(fh, "mode", "") != "rb" else io.TextIOWrapper(fh, encoding="utf-8")
            for line in text:
                if line.strip():
                    try: yield json.loads(line)
                    except Exception: continue
    finally:
        try: fh.close()
        except Exception: pass
print("engines ready")


In [ ]:
# Cell 4 — BUILD TRACKED: week units per (sub, period); exact-dedup capped; overflow fraction applied.
def week_bounds(s, e):
    out, cur = [], datetime.datetime.fromisoformat(s)
    end = datetime.datetime.fromisoformat(e)
    while cur < end:
        nxt = min(cur + datetime.timedelta(days=7), end)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d"))); cur = nxt
    return out

def to_unix(d): return int(datetime.datetime.fromisoformat(d).replace(tzinfo=datetime.timezone.utc).timestamp())

PDEF = [r for r in csv.DictReader(open(ROOT / "config/period_definitions.csv", encoding="utf-8")) if not r["model_id"].startswith("#")] if (ROOT / "config/period_definitions.csv").exists() else []
periods = [r for r in PDEF if r["sufficiency"] in ("sufficient", "axis_grade", "marginal_merge_first")]
if PERIOD_FILTER: periods = [r for r in periods if r["model_id"] in PERIOD_FILTER]
if SUB_FILTER: periods = [r for r in periods if r["subreddit_or_group"] == SUB_FILTER]
tracked = [r for r in periods if r["corpus_type"] == "tracked"] if BUILD_TRACKED else []
print(f"tracked periods to build: {len(tracked)}")

inspect_rows, built = [], []

def process_record(sub, ctype, pid, corpus, rec, seen, frac, shard, unit_id, counts):
    rid = str(rec.get("id", ""))
    if frac < 1.0 and (int(hashlib.sha256(rid.encode()).hexdigest(), 16) % 10000) >= frac * 10000:
        counts["sampled_out"] += 1; return
    raw = extract_text(ctype, rec)
    toks, fl = clean_and_tokenize(raw)
    if len(inspect_rows) < 200 and (fl["dropped"] or counts["n"] % 97 == 0):
        inspect_rows.append({"raw": raw[:300], "tokens": " ".join(toks[:40]), "kept": bool(toks), "reason": fl["dropped"] or "ok"})
    counts["n"] += 1
    if not toks:
        counts["drop_" + (fl["dropped"] or "x")] += 1; return
    lang, lmethod = lang_of(raw)
    if lang != "en":
        counts["non_en"] += 1; return
    key = hashlib.sha256(" ".join(toks).encode()).hexdigest()[:16]
    if key in seen: counts["dup"] += 1; return
    if len(seen) < 500000: seen.add(key)
    else: counts["dup_capped"] = 1
    cts = rec.get("created_utc", 0)
    try: iso = datetime.datetime.fromtimestamp(int(cts), datetime.timezone.utc).isoformat()
    except Exception: iso = ""
    shard.add({"rid_hash": hashlib.sha256(rid.encode()).hexdigest()[:16], "sub": sub, "ts": iso,
               "period": pid, "ctype": ctype[:3], "tokens": toks,
               "flags": {"lang": lang + ":" + lmethod, "bot": False}}, unit_id)
    counts["use"] += 1; counts["tok"] += len(toks)

t0 = time.time(); n_done = n_fail = 0
for pr in tracked:
    sub, pid, frac = pr["subreddit_or_group"], pr["model_id"], float(pr.get("sampling_fraction", 1.0) or 1.0)
    for ctype in ["comments", "submissions"]:
        for (ws, we) in week_bounds(pr["start_date"], pr["end_date"]):
            unit = f"shard__{sub}__{ctype}__{ws}_{we}"
            prior = {r["unit_id"]: r for r in rrows()}.get(unit)
            if prior and prior["status"] == "complete":
                # Validated skip: check all output shards exist and match recorded hash
                if verify_output_shards(prior.get("output_final", ""), prior.get("output_sha256", "")):
                    continue
            row = {"unit_id": unit, "source": "arctic_shift_api", "source_url_or_query": f"subreddit={sub}&{ws}..{we}&sort=asc",
                   "subreddit": sub, "start_ts": ws + "T00:00:00Z", "end_ts": we + "T00:00:00Z", "content_type": ctype,
                   "status": "in_progress", "attempt_count": int((prior or {}).get("attempt_count", 0)) + 1,
                   "started_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "completed_at": "",
                   "last_cursor": "", "n_read": 0, "n_usable": 0, "token_estimate": 0, "output_tmp": "",
                   "output_final": "", "output_sha256": "", "error_category": "", "error_excerpt": "",
                   "config_version": cfg["config_version"], "config_sha256": CFG_SHA, "retrieval_date": today}
            rupsert(row)
            try:
                if DRY_RUN:
                    import random; random.seed(11)
                    batches = [[{"id": f"t1_{i:05d}", "subreddit": sub, "created_utc": to_unix(ws) + i,
                                 "body": f"Dry run record {i} about community life, not perfect but not broken.",
                                 "title": f"Dry {i}", "selftext": "Research and teaching take time."} for i in range(120)]]
                else:
                    batches = api_pages(sub, ctype, to_unix(ws), to_unix(we), unit)
                counts = defaultdict(int); seen = set(); shard = None; outs = []
                def rot(sub=sub, pid=pid, ctype=ctype):
                    sid = f"tracked__{sub.lower()}__{pid.split('__')[-1]}__{len(built + outs):04d}"
                    return ShardWriter(sid, {"corpus": "tracked", "sub": sub, "period": pid, "ctype": ctype, "config": cfg["config_version"]})
                for batch in batches:
                    if shard is None: shard = rot()
                    for rec in batch:
                        row["last_cursor"] = str(rec.get("created_utc", ""))
                        process_record(sub, ctype, pid, "tracked", rec, seen, frac, shard, unit, counts)
                        if shard.n >= int(SL["records_per_shard"]):
                            outs.append(shard.close()); shard = rot()
                if shard is not None and shard.n:
                    outs.append(shard.close())
                for o in outs:
                    supsert({"shard_id": o["shard_id"], "corpus": "tracked", "subreddit": sub, "period_id": pid,
                             "content_type": ctype, "n_records": o["n_records"], "n_tokens": o["n_tokens"],
                             "min_ts": o["min_ts"], "max_ts": o["max_ts"], "path": o["path"], "sha256": o["sha256"],
                             "bytes_compressed": o["bytes_compressed"], "source_unit_ids": o["source_unit_ids"],
                             "status": "complete", "config_version": cfg["config_version"], "config_sha256": CFG_SHA,
                             "created_at": today})
                row.update({"status": "complete" if PAGES_PER_WEEK_CAP is None else "partial",
                            "completed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                            "n_read": counts["n"], "n_usable": counts["use"], "token_estimate": counts["tok"],
                            "output_final": ";".join(o["path"] for o in outs),
                            "output_sha256": ";".join(o["sha256"] for o in outs)})
                rupsert(row); built.extend(outs); n_done += 1
                print(f"OK {unit}: read={counts['n']} usable={counts['use']} tok={counts['tok']} shards={len(outs)}")
            except Exception as e:
                lg.exception(unit)
                row.update({"status": "failed", "error_category": "api", "error_excerpt": str(e)[:300]}); rupsert(row); n_fail += 1
                print(f"FAIL {unit}: {str(e)[:130]}")

print(f"tracked done={n_done} failed={n_fail} shards={len(built)} elapsed={time.time()-t0:.0f}s")

In [ ]:
# Cell 5 — BUILD REFERENCE: uniform stable-ID hash fraction per period (batched shards).
ref_periods = [r for r in periods if r["corpus_type"] == "reference"] if BUILD_REFERENCE else []
print(f"reference periods: {len(ref_periods)}")
ref_comp = []

for pr in ref_periods:
    pid, frac = pr["model_id"], float(pr.get("sampling_fraction", 1.0) or 1.0)
    per_sub = defaultdict(int)
    subs = sorted({r["subreddit_or_group"] for r in PDEF if r["corpus_type"] == "tracked"})
    
    counts = defaultdict(int); seen = set(); shard = None; outs = []
    def rot_ref(pid=pid):
        sid = f"reference__REF__{pid.split('__')[-1]}__{len(built + ref_comp + outs):04d}"
        return ShardWriter(sid, {"corpus": "reference", "period": pid, "config": cfg["config_version"]})

    for sub in subs:
        for ctype in ["comments", "submissions"]:
            for (ws, we) in week_bounds(pr["start_date"], pr["end_date"])[:4 if DRY_RUN else None]:
                unit = f"ref__{sub}__{ctype}__{ws}_{we}"
                prior = {r["unit_id"]: r for r in rrows()}.get(unit)
                if prior and prior["status"] == "complete":
                    if verify_output_shards(prior.get("output_final", ""), prior.get("output_sha256", "")):
                        continue
                try:
                    batches = [[{"id": f"t1_{i:05d}", "subreddit": sub, "created_utc": to_unix(ws) + i,
                                 "body": f"Reference dry record {i} across the platform, varied and ordinary."}] for i in range(1)] if DRY_RUN else api_pages(sub, ctype, to_unix(ws), to_unix(we), unit)
                    if shard is None: shard = rot_ref()
                    for batch in batches:
                        for rec in batch:
                            before = counts["use"]
                            process_record(sub, ctype, pid, "reference", rec, seen, frac, shard, unit, counts)
                            per_sub[sub] += counts["use"] - before
                            if shard.n >= int(SL["records_per_shard"]):
                                outs.append(shard.close()); shard = rot_ref()
                except Exception as e:
                    lg.error(f"ref {sub}/{ctype}/{ws}: {str(e)[:120]}")

    if shard is not None and shard.n:
        outs.append(shard.close())
        
    for o in outs:
        supsert({"shard_id": o["shard_id"], "corpus": "reference", "subreddit": "REF", "period_id": pid,
                 "content_type": "mixed", "n_records": o["n_records"], "n_tokens": o["n_tokens"],
                 "min_ts": o["min_ts"], "max_ts": o["max_ts"], "path": o["path"], "sha256": o["sha256"],
                 "bytes_compressed": o["bytes_compressed"], "source_unit_ids": o["source_unit_ids"],
                 "status": "complete", "config_version": cfg["config_version"], "config_sha256": CFG_SHA,
                 "created_at": today})
    ref_comp.extend(outs)
    tot = sum(per_sub.values())
    atomic = ROOT / "diagnostics/shards" / f"ref_composition__{pid.split('__')[-1]}.csv"
    atomic_text(atomic, "sub,docs,share\n" + "".join(f"{s},{c},{c/max(1,tot):.3f}\n" for s, c in sorted(per_sub.items(), key=lambda x: -x[1])))
    print(f"REF {pid}: docs={tot} subs={len(per_sub)} shards={len(outs)} (composition -> {atomic.name})")

In [ ]:
# Cell 6 — VALIDATE shards + inspection sample: reopen all, cross-check manifest, quarantine failures.
import glob as _glob
probs, checked = [], 0
for r in srows():
    if r["status"] != "complete": continue
    if PERIOD_FILTER and r["period_id"] not in PERIOD_FILTER: continue
    p = Path(r["path"])
    try:
        assert p.exists() and p.stat().st_size > 0, "missing/empty"
        with gzip.open(p, "rt", encoding="utf-8") as f:
            assert f.readline().startswith("#manifest"), "no header"
            n = t = 0
            for line in f:
                rec = json.loads(line)
                assert isinstance(rec.get("tokens"), list) and rec["tokens"], "bad tokens"
                assert "author" not in rec and "username" not in rec, "PII leak"
                n += 1; t += len(rec["tokens"])
        assert n == int(r["n_records"]) and t == int(r["n_tokens"]), "count mismatch"
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == r["sha256"], "checksum mismatch"
        checked += 1
    except Exception as e:
        probs.append((r["shard_id"], str(e)[:150]))
        q = ROOT / "shards/quarantine" / p.name
        try:
            if p.exists(): os.replace(p, q)
        except Exception: pass
        r2 = dict(r); r2["status"] = "validation_failed"; supsert(r2)
print(f"validated: {checked} shards; problems: {len(probs)}")
for sid, e in probs[:10]: print("  QUARANTINE", sid, e)
INSP = ROOT / "diagnostics/shards/shards_inspection_200.csv"
tmp = INSP.with_suffix(".tmp")
f = open(tmp, "w", newline="", encoding="utf-8")
w = csv.DictWriter(f, fieldnames=["raw", "tokens", "kept", "reason"]); w.writeheader()
w.writerows(inspect_rows[:200]); f.close(); os.replace(tmp, INSP)
print(f"inspection: {len(inspect_rows[:200])} rows -> {INSP} (read >=50 before training)")


In [ ]:
# Cell 7 — END-OF-RUN SUMMARY.
print("=" * 70)
print(f"SHARDS COMPLETE tracked_units={n_done} failed={n_fail} ref_shards={len(ref_comp)} validated={checked} quarantined={len(probs)}")
print("store : shards/tokenized/<corpus>__*.jsonl.gz + manifests/shard_manifest.csv (registry)")
print("rerun safe: YES (validated week-units skip by checksum; partial weeks reprocess idempotently)")
print("next  : 04_train_evaluate.ipynb (after reading the inspection sample)")
print("=" * 70)
